## 全体設計
```
動画 -> frame分解
  ↓
各frame:
  ・edge生成（ControlNet用）
  ・LoRAでスタイル付与
  ・ControlNetで構造固定
  ↓
frame結合 -> 動画
```

## 背景
- LoRAだけ -> スタイルは付くがブレる
- ControlNet -> 構造固定で安定
- 両方必須

## batch切り替え
- `USE_BATCH = True`  : 複数frameをまとめてpipeに渡す（高速）
- `USE_BATCH = False` : 1frameずつ処理（デバッグ向け）

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from pathlib import Path

from diffusers import (
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
)

import sys
sys.path.append('/workspace')
from src.utils.video_utility import show_before_after, write_video

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_pipe():
    device = 'cuda'
    dtype = torch.float16

    controlnet = ControlNetModel.from_pretrained(
        '/workspace/weights/controlnet-canny',
        torch_dtype=dtype,
    )

    pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        controlnet=controlnet,
        torch_dtype=dtype,
    ).to(device)

    pipe.load_lora_weights(
        '/workspace/weights/lora',
        weight_name='anime.safetensors',
    )
    pipe.set_adapters(['default_0'], adapter_weights=[0.8])
    pipe.safety_checker = None
    return pipe

In [3]:
def process_frame(pipe, frame, prompt):
    rgb = frame[:, :, ::-1]
    image = Image.fromarray(rgb)

    edge = cv2.Canny(frame, 100, 200)
    edge = np.stack([edge] * 3, axis=-1)
    edge_pil = Image.fromarray(edge)

    result = pipe(
        prompt=prompt,
        image=image,
        control_image=edge_pil,
        strength=0.5,
        guidance_scale=7.5,
        num_inference_steps=20,
    ).images[0]

    out = np.array(result)
    return out[:, :, ::-1]

In [4]:
def process_batch(pipe, frames, prompt):
    images = []
    edges = []

    for frame in frames:
        rgb = frame[:, :, ::-1]
        images.append(Image.fromarray(rgb))

        edge = cv2.Canny(frame, 100, 200)
        edge = np.stack([edge] * 3, axis=-1)
        edges.append(Image.fromarray(edge))

    B = len(images)
    results = pipe(
        prompt=[prompt] * B,
        image=images,
        control_image=edges,
        strength=0.5,
        guidance_scale=7.5,
        num_inference_steps=20,
    ).images

    out_frames = []
    for img in results:
        arr = np.array(img)
        out_frames.append(arr[:, :, ::-1])

    return out_frames

In [5]:
def process_video(in_path, out_path, prompt, use_batch=True, batch_size=3):
    pipe = build_pipe()

    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        raise RuntimeError(f'failed to open input video: {in_path}')

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_frames_all = []

    if use_batch:
        batch_frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            batch_frames.append(frame)
            if len(batch_frames) == batch_size:
                out_frames_all.extend(process_batch(pipe, batch_frames, prompt))
                batch_frames = []
        if batch_frames:
            out_frames_all.extend(process_batch(pipe, batch_frames, prompt))
    else:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            out_frames_all.append(process_frame(pipe, frame, prompt))

    cap.release()

    if not out_frames_all:
        raise RuntimeError(f'no output frames generated: {in_path}')

    write_video(out_path, out_frames_all, fps, w, h)


def process_video_batch(in_path, out_path, prompt, batch_size=3):
    return process_video(in_path, out_path, prompt, use_batch=True, batch_size=batch_size)


def process_video_nobatch(in_path, out_path, prompt):
    return process_video(in_path, out_path, prompt, use_batch=False)

In [6]:
WORKSPACE = Path('/workspace')
VIDEO_DIR = WORKSPACE / 'data' / 'videos'
OUT_DIR = WORKSPACE / 'logs' / 'notebook' / 'task_03_apply_style'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHOW_COMPARE = True
STYLE = 'oil_painting'
PROMPT = f'{STYLE} style'
USE_BATCH = True
BATCH_SIZE = 3

file_name_in = '94msufYZzaQ_26_0to273.mp4'
file_path_in = VIDEO_DIR / file_name_in
mode_name = 'batch' if USE_BATCH else 'nobatch'
file_path_out = OUT_DIR / f'processed_{STYLE}_{mode_name}_{file_name_in}'

if not file_path_in.exists():
    raise FileNotFoundError(f'input video not found: {file_path_in}')

process_video(
    str(file_path_in),
    str(file_path_out),
    prompt=PROMPT,
    use_batch=USE_BATCH,
    batch_size=BATCH_SIZE,
)

print('video      :', file_path_in.name)
print('action     :', 'apply_style')
print('style      :', STYLE)
print('prompt     :', PROMPT)
print('mode       :', mode_name)
print('batch_size :', BATCH_SIZE if USE_BATCH else '-')
print('out        :', file_path_out)

if SHOW_COMPARE:
    show_before_after(str(file_path_in), str(file_path_out))

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 4444.49it/s]3.10it/s]
StableDiffusionSafetyChecker LOAD REPORT from: /workspace/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 5472.17it/s]3.35it/s]
CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  

KeyboardInterrupt: 